# DermaScanAI — Skin Disease Classifier Training Notebook

## Step 0 — Install dependencies

In [ ]:
!pip install -q kaggle torch torchvision scikit-learn pandas pillow tqdm
print("Done.")

## Step 1 — Download the HAM10000 dataset

In [ ]:
import os

# Kaggle now provides an API token (string KGAT_...) instead of a kaggle.json file for downloads.
# If you still get a classic kaggle.json file, see the commented alternative below.

KAGGLE_TOKEN = input("Paste your Kaggle API token (KGAT_...): ").strip()

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/access_token", "w") as f:
    f.write(KAGGLE_TOKEN)
os.chmod("/root/.kaggle/access_token", 0o600)
print("Kaggle token has been set.")

# Alternative if you get a classic kaggle.json file instead of a token:
# from google.colab import files
# uploaded = files.upload()
# with open("/root/.kaggle/kaggle.json", "wb") as f:
#     f.write(list(uploaded.values())[0])
# os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [ ]:
!mkdir -p /content/data/ham10000
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data/ham10000 --unzip
print("HAM10000 downloaded.")

## Step 2 — Download the PAD-UFES-20 dataset

In [ ]:
import os, zipfile, glob, re

os.makedirs("/content/data/padufes", exist_ok=True)

!pip -q install gdown
import gdown

print("In Google Drive: right-click the zip -> Share -> 'Anyone with the link' -> Copy link")
drive_link = input("Paste the Google Drive link here: ").strip()

m = re.search(r"/d/([a-zA-Z0-9_-]+)", drive_link) or re.search(r"id=([a-zA-Z0-9_-]+)", drive_link)
assert m, "Can't extract the ID from the link - check that the full sharing link was copied correctly."
file_id = m.group(1)

dest_zip = "/content/data/padufes/padufes20_drive.zip"
gdown.download(id=file_id, output=dest_zip, quiet=False)
assert os.path.exists(dest_zip) and os.path.getsize(dest_zip) > 0, (
    "Download failed - check that access is really set to 'Anyone with the link'."
)
print("Downloaded:", dest_zip, os.path.getsize(dest_zip), "bytes")

# extract - repeat until no unpacked zip remains
# (some zips contain nested zips, e.g. imgs_part_1.zip inside the main zip)
while True:
    zips = glob.glob("/content/data/padufes/**/*.zip", recursive=True)
    if not zips:
        break
    progressed = False
    for zpath in zips:
        try:
            with zipfile.ZipFile(zpath, "r") as z:
                z.extractall(os.path.dirname(zpath))
            print(f"Extracted: {zpath}")
            os.remove(zpath)
            progressed = True
        except Exception as e:
            print(f"Skipped (not a valid zip): {zpath} -> {e}")
    if not progressed:
        break

print("PAD-UFES-20 ready.")

## Step 3 — Merge the two datasets

In [ ]:
import pandas as pd
import glob, os

# --- HAM10000 metadata ---
ham_csv_candidates = glob.glob("/content/data/ham10000/**/*metadata*.csv", recursive=True)
assert ham_csv_candidates, "HAM10000_metadata.csv not found - check that Step 1 completed successfully."
ham_meta = pd.read_csv(ham_csv_candidates[0])
print("HAM10000 metadata:", ham_meta.shape, "columns:", list(ham_meta.columns))

# index all images by filename (without extension) for fast lookup
ham_images = {}
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for p in glob.glob(f"/content/data/ham10000/**/{ext}", recursive=True):
        ham_images[os.path.splitext(os.path.basename(p))[0]] = p
print("Found HAM10000 images:", len(ham_images))

# --- PAD-UFES-20 metadata ---
pad_csv_candidates = [p for p in glob.glob("/content/data/padufes/**/*.csv", recursive=True)]
assert pad_csv_candidates, "metadata.csv for PAD-UFES-20 not found - check that Step 2 completed successfully."
pad_meta = pd.read_csv(pad_csv_candidates[0])
print("PAD-UFES-20 metadata:", pad_meta.shape, "columns:", list(pad_meta.columns))

pad_images = {}
for ext in ("*.png", "*.jpg", "*.jpeg"):
    for p in glob.glob(f"/content/data/padufes/**/{ext}", recursive=True):
        pad_images[os.path.splitext(os.path.basename(p))[0]] = p
print("Found PAD-UFES-20 images:", len(pad_images))

In [ ]:
# --- mapping to unified categories ---

HAM10000_MAP = {
    "akiec": "actinic_keratosis",
    "bcc":   "basal_cell_carcinoma",
    "bkl":   "seborrheic_keratosis",
    "df":    "dermatofibroma",
    "mel":   "melanoma",
    "nv":    "nevus",
    "vasc":  "vascular_lesion",
}

PAD_UFES_MAP = {
    "ACK": "actinic_keratosis",
    "BCC": "basal_cell_carcinoma",
    "SCC": "squamous_cell_carcinoma",
    "BOD": "squamous_cell_carcinoma",  # Bowen's disease = SCC in situ
    "SEK": "seborrheic_keratosis",
    "MEL": "melanoma",
    "NEV": "nevus",
}

records = []

for _, row in ham_meta.iterrows():
    dx = str(row.get("dx", "")).strip().lower()
    label = HAM10000_MAP.get(dx)
    img_id = str(row.get("image_id", "")).strip()
    path = ham_images.get(img_id)
    if label and path:
        records.append({"path": path, "label": label, "source": "ham10000"})

# find the diagnosis and id columns in PAD-UFES-20 (names vary by version)
diag_col = next((c for c in pad_meta.columns if c.lower() in ("diagnostic", "diagnosis", "label")), None)
id_col = next((c for c in pad_meta.columns if "img" in c.lower() and "id" in c.lower()), None)
assert diag_col and id_col, f"Check the columns in the PAD-UFES-20 CSV: {list(pad_meta.columns)}"

for _, row in pad_meta.iterrows():
    dx = str(row.get(diag_col, "")).strip().upper()
    label = PAD_UFES_MAP.get(dx)
    img_id = str(row.get(id_col, "")).strip()
    img_id_noext = os.path.splitext(img_id)[0]
    path = pad_images.get(img_id_noext) or pad_images.get(img_id)
    if label and path:
        records.append({"path": path, "label": label, "source": "padufes20"})

df = pd.DataFrame(records)
print("Total images after merging:", len(df))
print(df.groupby(["label", "source"]).size().unstack(fill_value=0))

UNIFIED_CLASSES = sorted(df["label"].unique().tolist())
print("\nCategories:", UNIFIED_CLASSES)

## Step 4 — Train/validation split and data preparation

In [ ]:
from sklearn.model_selection import train_test_split
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np

label_to_idx = {label: i for i, label in enumerate(UNIFIED_CLASSES)}
df["label_idx"] = df["label"].map(label_to_idx)

train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df["label_idx"], random_state=42
)
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")

IMAGE_SIZE = 244

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        image = self.transform(image)
        return image, row["label_idx"]


train_dataset = SkinLesionDataset(train_df, train_transform)
val_dataset = SkinLesionDataset(val_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

# class weights - HAM10000/PAD-UFES-20 are heavily imbalanced (nevus dominates)
class_counts = train_df["label_idx"].value_counts().sort_index()
class_weights = (1.0 / class_counts).values
class_weights = class_weights / class_weights.sum() * len(UNIFIED_CLASSES)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", dict(zip(UNIFIED_CLASSES, class_weights.tolist())))

## Step 5 — Model (EfficientNet-B3, transfer learning)

In [ ]:
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def build_model(num_classes):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_model(len(UNIFIED_CLASSES)).to(device)
print("Model ready with", len(UNIFIED_CLASSES), "categories.")

## Step 6 — Training

In [ ]:
import torch.optim as optim
from sklearn.metrics import f1_score
import copy, os

EPOCHS = 15
CHECKPOINT_PATH = "/content/checkpoint_best.pt"

# try to mount Drive for checkpointing - if it succeeds, training
# survives disconnects/runtime resets (resumes from the last best model)
DRIVE_CHECKPOINT_DIR = None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/skinscan_checkpoints"
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    print("Checkpoint will also be saved to Google Drive:", DRIVE_CHECKPOINT_DIR)
except Exception as e:
    print("Drive is not available - checkpoint will be saved locally in Colab only:", e)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

start_epoch = 0
best_f1 = 0.0
best_state = None

# if a checkpoint from a previous attempt already exists, resume from there instead of starting over
resume_path = None
if DRIVE_CHECKPOINT_DIR and os.path.exists(f"{DRIVE_CHECKPOINT_DIR}/checkpoint_best.pt"):
    resume_path = f"{DRIVE_CHECKPOINT_DIR}/checkpoint_best.pt"
elif os.path.exists(CHECKPOINT_PATH):
    resume_path = CHECKPOINT_PATH

if resume_path:
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    best_f1 = ckpt.get("best_f1", 0.0)
    start_epoch = ckpt.get("epoch", 0)
    best_state = copy.deepcopy(model.state_dict())
    print(f"Resuming from a previous checkpoint: after epoch {start_epoch}, best_f1={best_f1:.4f}")

def save_checkpoint(epoch, state_dict, f1):
    payload = {"state_dict": state_dict, "best_f1": f1, "epoch": epoch, "class_names": UNIFIED_CLASSES}
    torch.save(payload, CHECKPOINT_PATH)
    if DRIVE_CHECKPOINT_DIR:
        try:
            torch.save(payload, f"{DRIVE_CHECKPOINT_DIR}/checkpoint_best.pt")
        except Exception as e:
            print("Failed to save to Drive:", e)

for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_dataset)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | val_macro_f1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())
        save_checkpoint(epoch + 1, best_state, best_f1)
        print(f"  -> new best model (f1={best_f1:.4f}) - checkpoint saved")

model.load_state_dict(best_state)
print(f"\nTraining finished. Best val macro F1: {best_f1:.4f}")

## Step 7 — Results

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=UNIFIED_CLASSES))
print("Confusion matrix (rows=actual, columns=predicted):")
print(pd.DataFrame(confusion_matrix(all_labels, all_preds), index=UNIFIED_CLASSES, columns=UNIFIED_CLASSES))

## Step 8 — Save and download the model

In [ ]:
import json, os

DISPLAY_INFO = {
    "melanoma": {"display_name": "Melanoma", "severity": "HIGH"},
    "basal_cell_carcinoma": {"display_name": "Basal Cell Carcinoma", "severity": "HIGH"},
    "squamous_cell_carcinoma": {"display_name": "Squamous Cell Carcinoma", "severity": "HIGH"},
    "actinic_keratosis": {"display_name": "Actinic Keratosis", "severity": "MEDIUM"},
    "seborrheic_keratosis": {"display_name": "Seborrheic Keratosis", "severity": "LOW"},
    "nevus": {"display_name": "Nevus (Mole)", "severity": "LOW"},
    "dermatofibroma": {"display_name": "Dermatofibroma", "severity": "LOW"},
    "vascular_lesion": {"display_name": "Vascular Lesion", "severity": "LOW"},
}

os.makedirs("/content/output", exist_ok=True)

torch.save(
    {"state_dict": model.state_dict(), "class_names": UNIFIED_CLASSES},
    "/content/output/skin_model.pt",
)

label_converter = {
    label: DISPLAY_INFO.get(label, {"display_name": label.replace("_", " ").title(), "severity": "MEDIUM"})
    for label in UNIFIED_CLASSES
}
with open("/content/output/label_converter.json", "w", encoding="utf-8") as f:
    json.dump(label_converter, f, indent=2, ensure_ascii=False)

print("Saved:")
print(" - /content/output/skin_model.pt")
print(" - /content/output/label_converter.json")

In [ ]:
from google.colab import files

files.download("/content/output/skin_model.pt")
files.download("/content/output/label_converter.json")